# IBM Granite in ShieldZ: explanations without exposure

After a payment is blocked or held, the customer gets a short explanation. ShieldZ uses **IBM Granite 4.0 (350 million parameters)** for the English wording, running on the LinuxONE VM through llama.cpp. This notebook loads the real model and walks through the whole path: the prompt, the generation, the guards around it, and what happens when the model misbehaves.

The design rule is that the model never sees personal data, and never gets the last word on what the customer should do.

1. The model on the platform
2. The prompt: fixed text and reason codes only
3. One generation, step by step
4. The guards, tested with bad outputs
5. Speed and the cache
6. Quality across many different prompts
7. Languages

In [ ]:
import io, json, os, platform, random, re, sys, tempfile, time, warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.figsize": (7, 3.2), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "figure.dpi": 100})

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "shieldz").exists())
sys.path[:0] = [str(REPO / "src"), str(REPO / "scripts")]
WORK = Path(tempfile.mkdtemp(prefix="shieldz_granite_"))
os.environ["SHIELDZ_ARTIFACTS"] = str(WORK)

import llama_cpp
weights = Path(os.environ.get("SHIELDZ_GRANITE_PATH", "/home/ibm-user/granite-4.0-h-350m-base-be.f16.gguf"))
print("Machine architecture :", platform.machine())
print("CPU cores visible    :", os.cpu_count())
print("llama.cpp (python)   :", llama_cpp.__version__)
print("Model file           :", weights.name)
print(f"Size on disk         : {weights.stat().st_size / 1e9:.2f} GB")

## 1. The model on the platform

The file name says what it is: `granite-4.0-h-350m-base-be.f16`. A **350M-parameter base model** (not an instruction-tuned chat model), in **big-endian** byte order (`be`, the native order of s390x), with 16-bit floats. It runs on the CPU. We did not use or measure the on-chip AI accelerator.

In [ ]:
from shieldz.explainer.granite import GraniteGenerator

t0 = time.time()
llm = GraniteGenerator()
print(f"Loaded in {time.time() - t0:.1f} s | threads: {llm.n_threads} | context window: 512 tokens")
meta = getattr(llm.llm, "metadata", {}) or {}
print({k: meta[k] for k in ("general.name", "general.architecture", "general.size_label") if k in meta})

## 2. The prompt: fixed text and reason codes only

The explainer receives a job that holds **only reason codes** from a closed list, a decision and a language. The prompt is fixed text, two fixed examples, and the English template sentence chosen by those codes. It contains no names, no accounts, no amounts, and no digits at all. A guard re-renders the prompt from the job and rejects anything that is not byte-for-byte identical.

In [ ]:
from shieldz.contracts import DecisionKind, ExplainJob, Language, Level, Reason, ReasonCode
from shieldz.explainer.guard import compose_explanation, validate_output, validate_prompt, validate_quality
from shieldz.explainer.prompt import build_prompt
from shieldz.explainer.template import FOOTER_EN, TemplateExplainer

job = ExplainJob(
    decision_id="0" * 32, decision=DecisionKind.BLOCK, risk_level=Level.HIGH, language=Language.EN,
    reasons=[Reason(code=ReasonCode.AMOUNT_HIGH_VS_BALANCE, level=Level.HIGH),
             Reason(code=ReasonCode.MULE_ACCOUNT_PATTERN, level=Level.HIGH),
             Reason(code=ReasonCode.ACCOUNT_DRAIN, level=Level.HIGH)],
)
prompt = build_prompt(job)
print(prompt)
print("\n" + "-" * 70)
print("digits in the prompt   :", sum(c.isdigit() for c in prompt))
print("prompt size            :", len(llm.llm.tokenize(prompt.encode())), "tokens (the model's window is 512)")
print("guard on this prompt   :", validate_prompt(prompt, job))
tampered = prompt.replace("Simple words:", "Payer Name: Ramesh Kulkarni\nSimple words:")
print("guard on a prompt with an injected name:", validate_prompt(tampered, job))

## 3. One generation, step by step

Granite continues the text after `Simple words:`. We take only its **explanation** of why the payment looks risky. Anything it says about *what the customer should do* is dropped, and the reviewed action sentence is always appended. In our tests the model told customers to call "their number" or "the number they give you". In a digital-arrest scam, that number belongs to the scammer.

In [ ]:
t0 = time.time()
raw = llm.generate(prompt)
generation_s = time.time() - t0
stored = compose_explanation(raw, FOOTER_EN)
kept = stored[: -len(FOOTER_EN)].strip() if stored else ""
dropped = raw.replace(kept, "").strip()
print(f"Generated in {generation_s:.1f} s (56 tokens at most, temperature 0, so the same prompt gives the same text)\n")
pd.DataFrame([
    ["1. What the model wrote", raw],
    ["2. Kept: the explanation of why", kept or "(nothing)"],
    ["3. Dropped: the model's own advice", dropped or "(nothing)"],
    ["4. Added: the reviewed action sentence", FOOTER_EN],
    ["5. What the customer sees", stored or "(rejected: the template is used)"],
], columns=["step", "text"])

In [ ]:
print("output guard  (personal-data shapes):", validate_output(stored) if stored else "n/a")
print("quality guard (loops, contradictions, advice):", validate_quality(stored) if stored else "n/a")
print()
print("The reviewed template for the same reason codes, for comparison:")
print(TemplateExplainer().explain(job).text)

## 4. The guards, tested with bad outputs

A small base model will sometimes loop, contradict the decision, state something false, or put a number in. These are real failure modes from our tests. The last row is the real output from step 3. A rejected output is replaced by the reviewed template, so the customer always gets safe text.

In [ ]:
def verdict(text):
    kept_and_action = compose_explanation(text, FOOTER_EN)
    if kept_and_action is None:
        return "rejected: nothing explanatory left after the model's advice is removed"
    ok, why = validate_output(kept_and_action)
    if not ok:
        return "rejected: " + why
    ok, why = validate_quality(kept_and_action)
    return "accepted" if ok else "rejected: " + why

cases = {
    "loops": "Do not ask for any information. Do not ask for any information. Do not ask for any information.",
    "contradicts the decision": "There is no evidence of fraud here. Please call your bank.",
    "states something false": "Your account has been drained. Please call your bank.",
    "only gives advice": "Please call the number they give you.",
    "puts a long number in": "The receiving account number is 123456789012. Please call your bank.",
    "the real output from step 3": raw,
}
pd.DataFrame([{"model output": text[:80] + ("..." if len(text) > 80 else ""), "case": name, "verdict": verdict(text)}
              for name, text in cases.items()])

## 5. Speed and the cache

Granite on three CPU cores takes about 10 seconds for a fresh explanation. That is far too slow to sit in the payment path, which is why the explainer is a separate container: the decision returns in milliseconds, the reviewed template text is shown at once, and Granite's wording replaces it a few seconds later.

The prompt contains only reason codes, so the text for a given combination of codes can be reused. The first payment with those codes pays the ten seconds, and the next one with the same codes does not. **These texts are generated once and cached, not freshly written for every payment.**

In [ ]:
from shieldz.explainer.journal import PromptJournal, load_journal
from shieldz.explainer.worker import Worker
from shieldz.outbox.queue import Outbox

box = Outbox(WORK / "outbox.sqlite")
worker = Worker(box, llm, PromptJournal(WORK / "journal.jsonl"))
rows = []
for i in (1, 2):
    j = job.model_copy(update={"decision_id": f"{i:032x}"})   # two different payments, same reason codes
    t0 = time.perf_counter()
    e = worker.process(j)
    rows.append({"payment": f"#{i}", "seconds": round(time.perf_counter() - t0, 2), "source": e.source.value,
                 "text": e.text[:95] + "..."})
display(pd.DataFrame(rows))
print("rows in the text cache :", box._conn().execute("SELECT COUNT(*) FROM llm_cache").fetchone()[0])
print("prompt journal entries :", len(load_journal(WORK / "journal.jsonl")), "(every prompt the model saw, with no personal data)")

## 6. Quality across many different prompts

One example proves little. Here are fresh generations for different combinations of reason codes (cache not used), each passed through the same guards. Then the committed run of 30 prompts.

In [ ]:
import granite_quality as gq

N_LIVE = int(os.environ.get("SHIELDZ_NB_PROMPTS", "10"))
rows = []
for chosen in gq.combos(random.Random(7), N_LIVE):
    decision = DecisionKind.BLOCK if len(chosen) >= 2 else DecisionKind.REVIEW
    j = ExplainJob(decision_id="0" * 32, decision=decision, language=Language.EN,
                   risk_level=Level.HIGH if decision == DecisionKind.BLOCK else Level.MEDIUM,
                   reasons=[Reason(code=c, level=Level.HIGH) for c in chosen])
    t0 = time.time()
    out = llm.generate(build_prompt(j))
    seconds = time.time() - t0
    rows.append({"reason codes": ", ".join(c.value for c in chosen), "seconds": round(seconds, 1),
                 "verdict": verdict(out), "model wrote": out[:70] + "..."})
live = pd.DataFrame(rows)
live

In [ ]:
accepted = live["verdict"].eq("accepted")
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.bar(range(len(live)), live["seconds"], color=["black" if a else "lightgray" for a in accepted])
ax.set_xlabel("prompt (a different combination of reason codes each)"); ax.set_ylabel("seconds to generate")
ax.set_title(f"{int(accepted.sum())} of {len(live)} accepted (black); the others fall back to the template (gray)")
plt.show()

In [ ]:
# The committed run of 30 distinct prompts, measured on the VM (results/granite_quality.md)
text = (REPO / "results" / "granite_quality.md").read_text()
display(Markdown("\n".join(text.splitlines()[2:6])))
runs = re.findall(r"^### (.+?)\s+\((PASS|FAIL.*?), ([\d.]+) s\)$", text, re.M)
stored = pd.DataFrame(runs, columns=["reason codes", "verdict", "seconds"])
stored["seconds"] = stored["seconds"].astype(float)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
axes[0].hist(stored["seconds"], bins=12, color="black")
axes[0].set_xlabel("seconds to generate"); axes[0].set_ylabel("prompts"); axes[0].set_title("Generation time, 30 prompts")
counts = stored["verdict"].str.startswith("PASS").value_counts()
axes[1].bar(["accepted", "template instead"], [counts.get(True, 0), counts.get(False, 0)], color=["black", "lightgray"])
axes[1].set_title("Passed the guards"); axes[1].set_ylabel("prompts")
plt.tight_layout(); plt.show()

## 7. Languages

A 350M base model answers in English whatever language it is asked for, so asking it for Hindi or Tamil would give the customer an English text under a Hindi label. English jobs go to Granite. Hindi and Tamil use **human-written templates**, the same reviewed sentences, instantly.

In [ ]:
rows = []
for lang in (Language.EN, Language.HI, Language.TA):
    j = job.model_copy(update={"language": lang})
    rows.append({"language": lang.value, "written by": "Granite (then guarded)" if lang == Language.EN else "reviewed template",
                 "text": TemplateExplainer().explain(j).text})
pd.DataFrame(rows)

## What this shows, and what it does not

**It shows** a real 350M Granite model running on s390x, a prompt that contains no personal data, and a pipeline in which the model's wording is checked, its advice is replaced by a reviewed sentence, and any bad output falls back to a reviewed template.

**It does not show** that Granite writes every explanation live (most are cached), that it writes Hindi or Tamil (it does not), or that it uses the platform's AI accelerator (it runs on the CPU). The explanations are short, and a small model's grammar is imperfect. That is why the action sentence and the languages other than English are reviewed text.

In [ ]:
import shutil
shutil.rmtree(WORK, ignore_errors=True)
print("scratch folder removed")